# NEXUS: Intelligent E-Commerce Product Recommendation Engine
### End-to-End Methodological Walkthrough, Validation & Evaluation
**Assigned Problem:** E-Commerce Product Recommendation Engine (Top-N Personalization)
**System Identity:** *Discover Beyond Popularity*
**Dataset:** UCI Online Retail Benchmark (541k transactions, CC BY 4.0)
---
This notebook demonstrates the end-to-end data pipeline, leakage-free chronological splitting, model training, ranking evaluation, and our two signature features (Serendipity Mode & Reasoning Lab).

In [1]:
import sys, os
sys.path.append('..')
import pandas as pd
import numpy as np
import json
from pathlib import Path

print('Python scientific environment successfully initialized.')

Python scientific environment successfully initialized.


## 1. Data Ingestion, Hygiene & Schema Audit
We load the processed merchandise catalog and clean transaction interactions.

In [2]:
catalog = pd.read_csv('../data/processed/catalog.csv', dtype={'StockCode': str})
transactions = pd.read_csv('../data/processed/clean_transactions.csv', nrows=10000, dtype={'StockCode': str, 'CustomerID': str})
print(f'Catalog items: {len(catalog):,} across {catalog["Category"].nunique()} categories')
transactions[['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'UnitPrice', 'CustomerID', 'Category']].head()

Catalog items: 3,913 across 9 categories


InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID,Category
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2.55,17850,Home Decor & Lighting
536365,71053,WHITE METAL LANTERN,6,3.39,17850,Home Decor & Lighting
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2.75,17850,Home Decor & Lighting
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,3.39,17850,Kitchen & Dining
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,3.39,17850,Home Decor & Lighting


## 2. Statistical Profiling & User-Item Matrix Sparsity
Real-world e-commerce matrices are extremely sparse. We examine matrix sparsity and popularity inequality.

In [3]:
with open('../reports/eda_summary.json', 'r') as f:
    eda = json.load(f)

print(f'Matrix Sparsity: {eda["user_item_matrix_sparsity_pct"]:.2f}%')
print(f'Popularity Gini Coefficient: {eda["item_popularity"]["gini_coefficient"]:.4f}')
print(f'Top 20% products capture: {eda["item_popularity"]["sales_share_top_20_pct_items"]:.2f}% of total transactions')

Matrix Sparsity: 98.32%
Popularity Gini Coefficient: 0.6378
Top 20% products capture: 65.57% of total transactions


## 3. Leakage-Free Chronological Splitting
To prevent future data leakage, interactions are ordered strictly by time.

In [4]:
with open('../data/processed/split_metadata.json', 'r') as f:
    split_meta = json.load(f)

print('Split Timestamps:')
print('  Train window:', split_meta['timestamps']['dataset_start'], '->', split_meta['timestamps']['train_end_val_start'])
print('  Test window: ', split_meta['timestamps']['val_end_test_start'], '->', split_meta['timestamps']['dataset_end'])
print(f'Warm evaluation users: {split_meta["user_cohorts"]["warm_eval_users_test"]}')
print(f'Cold-start new users:  {split_meta["user_cohorts"]["cold_start_users_test"]}')

Split Timestamps:
  Train window: 2010-12-01 08:26:00 -> 2011-11-02 12:26:00
  Test window:  2011-11-21 09:11:00 -> 2011-12-09 12:50:00
Warm evaluation users: 977
Cold-start new users:  170


## 4. Empirical Model Benchmark on Held-Out Test Data
Ranking evaluation comparing Popularity Baseline, Content-Based, Collaborative Filtering, and Adaptive Hybrid.

In [5]:
with open('../reports/model_comparison.json', 'r') as f:
    comp = json.load(f)

rows = []
for m_name, m_dict in comp['models_evaluated'].items():
    k10 = m_dict['K=10']
    rows.append({
        'Model': m_name,
        'Precision@10': k10['Precision'],
        'Recall@10': k10['Recall'],
        'NDCG@10': k10['NDCG'],
        'Catalog Coverage (%)': f"{k10['CatalogCoverage_Pct']:.2f}%",
        'Category Diversity': k10['CategoryDiversity'],
        'Novelty (bits)': k10['Novelty_Bits']
    })
pd.DataFrame(rows)

Model,Precision@10,Recall@10,NDCG@10,Catalog Coverage (%),Category Diversity,Novelty (bits)
Popularity Baseline,0.0703,0.0284,0.0799,0.26%,0.6889,8.1445
Content-Based (TF-IDF),0.0303,0.0202,0.0322,18.58%,0.2529,12.6814
Collaborative (SVD),0.0427,0.0208,0.0486,36.29%,0.6527,11.7957
Adaptive Hybrid,0.0437,0.0246,0.0499,34.68%,0.5751,11.5779


## 5. Live Inference with Signature Feature 1: Serendipity Mode
We test the 3 discovery strategies (Familiar, Balanced, Explore) for Customer 17850.

In [6]:
from src.recommendation.engine import NexusRecommenderEngine
engine = NexusRecommenderEngine(artifacts_dir='../artifacts', catalog_csv='../data/processed/catalog.csv', train_csv='../data/processed/train.csv')

for mode in ['familiar', 'balanced', 'explore']:
    res = engine.recommend(user_id='17850', serendipity_mode=mode, k=3)
    print(f'\n=== Strategy: {mode.upper()} ===')
    for r in res['recommendations']:
        print(f"  #{r['item_id']} | {r['description']} | Cat: {r['category']} | Score: {r['composite_score']} | Nov: {r['novelty_score']}")

2026-11-15 22:21:26,047 [INFO] NEXUS.Engine: Initializing NEXUS Recommender Engine...
2026-11-15 22:21:26,048 [INFO] NEXUS.PopularityModel: Loading PopularityRecommender from: ..\artifacts\popularity_model.joblib
2026-11-15 22:21:26,080 [INFO] NEXUS.ContentModel: Loading ContentBasedRecommender from: ..\artifacts\content_model.joblib
2026-11-15 22:21:27,151 [INFO] NEXUS.CollaborativeModel: Loading CollaborativeRecommender from: ..\artifacts\collaborative_model.joblib
2026-11-15 22:21:27,816 [INFO] NEXUS.HybridModel: Loading HybridRecommender from: ..\artifacts\hybrid_model.joblib
2026-11-15 22:21:34,153 [INFO] NEXUS.ReasoningLab: Indexing user transaction histories for Reasoning Lab...
2026-11-15 22:21:35,566 [INFO] NEXUS.Engine: NEXUS Recommender Engine initialized successfully and ready for inference.

=== Strategy: FAMILIAR ===
  #71038 | WHITE HANGING BEADS CANDLE HOLDER | Cat: Home Decor & Lighting | Score: 0.7049 | Nov: 0.4932
  #82484 | WOOD BLACK BOARD ANT WHITE FINISH | Cat: G

## 6. Signature Feature 2: Recommendation Reasoning Lab
We audit the grounded signals (collaborative, content, category, popularity) for a recommended product.

In [7]:
res = engine.recommend(user_id='17850', k=1, explain=True)
audit = res['recommendations'][0]['reasoning_audit']
print(f"Product: {audit['product_name']} (Category: {audit['category']})")
print('Ground Truth Evidence:')
for sig, data in audit['signals'].items():
    print(f"  [{sig.upper()}]: {data['detail']}")

Product: PEARL & SHELL 42"NECKL. IVORY (Category: Gifts & Novelties)
Ground Truth Evidence:
  [COLLABORATIVE_FILTERING]: Shoppers who purchased 'WHITE HANGING HEART T-LIGHT HOLDER' frequently bought this item (Latent Factor Cosine Affinity: 0.56).
  [CONTENT_SEMANTICS]: Semantic overlap on key terms ['NOVELTIES', 'GIFTS NOVELTIES', 'GIFTS', 'IVORY'] matching your historical purchase descriptions (TF-IDF Cosine Similarity: 0.0731).
  [CATEGORY_ALIGNMENT]: You have purchased 463 items in 'Gifts & Novelties' (26.7% of your shopping history).
  [CATALOG_POPULARITY]: Catalog popularity rank #3460 (Long-Tail Discovery, Top 95.8% of catalog).


## 7. Cold-Start Fallback Demonstration
We test an unseen new customer with zero interaction history.

In [8]:
cold_res = engine.recommend(user_id='NEW_USER_99999', k=3)
print(f"User Status: {cold_res['user_status']}")
for r in cold_res['recommendations']:
    print(f"  {r['item_id']} | {r['description']} | Fallback: {r.get('is_fallback')}")

User Status: COLD_START
  22423 | REGENCY CAKESTAND 3 TIER | Fallback: True
  85123A | WHITE HANGING HEART T-LIGHT HOLDER | Fallback: True
  47566 | PARTY BUNTING | Fallback: True
